In [1]:
a = [1, 2]
b = a
b.append(3)

print(a)        # [1, 2, 3]
print(a is b)   # True：指向同一个对象
print(a == b)   # True：值相等


[1, 2, 3]
True
True


In [4]:
def mutate(xs: list[int]) -> None:
    xs.append(1)          # 修改调用方可见的同一个列表

def rebind(xs: list[int]) -> None:
    print("参数xs地址情况",id(xs))
    xs = [1,2]              # 只让局部名字 xs 指向新列表
    print("rebind function内部赋值给xs后，xs地址情况",id(xs))

data: list[int] = []
mutate(data)
print(data)               # []变成了[1]

print(id(data))
rebind(data)
print(data)               # 仍然是 [1]


[1]
4567646976
参数xs地址情况 4567646976
rebind function内部赋值给xs后，xs地址情况 4567645120
[1]


In [7]:
# 错误：同一个列表会被多次调用共享
def collect_bad(x: int, result: list[int] = []) -> list[int]:
    result.append(x)
    return result

# 正确：用 None 表示“未提供”
def collect(x: int, result: list[int] | None = None) -> list[int]:
    if result is None:
        result = []
    result.append(x)
    return result
print(collect_bad(1))
collect_bad(2), collect(1),collect(2)

[1]


([1, 2], [1], [2])

In [8]:
import copy

src = [[1], [2]]
shallow = src.copy()           # 只复制最外层列表
deep = copy.deepcopy(src)      # 递归复制内部对象

shallow[0].append(9)
print(src)                     # [[1, 9], [2]]
print(deep)                    # [[1], [2]]

[[1, 9], [2]]
[[1], [2]]


In [11]:
from collections.abc import Callable
# local enclosure global builtin. Python 按 LEGB 顺序查找名字
def make_multiplier(scale: float) -> Callable[[float], float]:
    def multiply(x: float) -> float:
        return x * scale # 闭包会保存外层作用域的变量
    return multiply

double = make_multiplier(2.0)
print(double(3.0))  # 6.0

6.0


In [15]:
# 三个函数最终都读取同一个 i，结果都是 2
bad = [lambda: i for i in range(3)] # 循环中创建闭包时要注意延迟绑定：闭包在调用时才读取外层名字。
print([fn() for fn in bad])


# 用默认参数在每轮定义时保存当前值
good = [lambda i=i: i for i in range(3)] # 默认参数会在函数创建时立即求值
print([fn() for fn in good])  # [0, 1, 2]


[2, 2, 2]
[0, 1, 2]


In [16]:
from dataclasses import dataclass
from collections.abc import Iterator

@dataclass
class Batch:
    samples: list[list[float]]

    def __len__(self) -> int:
        return len(self.samples)

    def __iter__(self) -> Iterator[list[float]]:
        return iter(self.samples)

batch = Batch([[1.0, 2.0], [3.0, 4.0]])
print(len(batch))
for sample in batch:
    print(sample)


2
[1.0, 2.0]
[3.0, 4.0]


In [17]:
from collections.abc import Callable
from functools import wraps
from time import perf_counter
from typing import ParamSpec, TypeVar

P = ParamSpec("P")
R = TypeVar("R")

def timed(fn: Callable[P, R]) -> Callable[P, R]:
    @wraps(fn)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        start = perf_counter()
        try:
            return fn(*args, **kwargs)
        finally:
            elapsed_ms = (perf_counter() - start) * 1_000
            print(f"{fn.__name__}: {elapsed_ms:.3f} ms")
    return wrapper

@timed
def preprocess(values: list[float]) -> list[float]:
    return [value * 2 for value in values]


In [22]:
from contextlib import contextmanager
from collections.abc import Iterator
from time import perf_counter

@contextmanager
def timer(name: str) -> Iterator[None]:
    start = perf_counter()
    try:
        yield
    finally:
        elapsed_ms = (perf_counter() - start) * 1_000
        print(f"{name}: {elapsed_ms:.3f} ms")

with timer("load data"):
    data = [value for value in range(100_000)]


load data: 3.883 ms


In [34]:
def get_number(text):
    try:
        return int(text)
    except ValueError as exc:
        raise RuntimeError("输入不是数字") from exc
get_number("abc")

RuntimeError: 输入不是数字

In [35]:
def get_number(text):
    try:
        return int(text)
    except ValueError as exc:
        raise RuntimeError("输入不是数字")
get_number("abc")


RuntimeError: 输入不是数字

In [43]:
# 从一次正常的asyncio出发
import asyncio
import time

async def worker(name: str) -> str:
    # time.sleep(2)   # time: 4.0060060024261475
    await asyncio.sleep(2) # time: 2.0027430057525635
    return f"{name} done"

async def main():
    results = await asyncio.gather(
        worker("a"),
        worker("b"),
    )
    print(results)
st = time.time()
await main()
print("time:",time.time()-st)

['a done', 'b done']
time: 2.0027430057525635


In [41]:
# 正确处理asycnio中的blocking work, 阻塞io用to_thread（如果无法换成非阻塞io）
import asyncio
import time

def blocking_work(name: str) -> str:
    time.sleep(2)
    return f"{name} done"

async def main():
    results = await asyncio.gather(
        asyncio.to_thread(blocking_work, "A"),
        asyncio.to_thread(blocking_work, "B"),
    )
    print(results)
st = time.time()
await main()
print("time:",time.time()-st)

['A done', 'B done']
time: 2.006127119064331


In [47]:
N = 5 * 10**7

def cpu_work(n: int) -> int:
    total = 0
    for i in range(n):
        total += i * i
    return total
st = time.time()
cpu_work(N)
print("time:",time.time()-st)

time: 1.347869873046875


In [48]:
# 正确处理asycnio中的blocking work, 阻塞io用to_thread（如果无法换成非阻塞io）
import asyncio
import time

N = 5 * 10**7

def cpu_work(n: int) -> int:
    total = 0
    for i in range(n):
        total += i * i
    return total

def blocking_work(name: str) -> str:
    time.sleep(2)
    return f"{name} done"

async def main():
    results = await asyncio.gather(
        asyncio.to_thread(cpu_work, N),
        asyncio.to_thread(cpu_work, N),
    )
    print(results)
st = time.time()
await main()
print("time:",time.time()-st)

[41666665416666675000000, 41666665416666675000000]
time: 2.5123438835144043


In [49]:
import asyncio, time, multiprocessing
from concurrent.futures import ProcessPoolExecutor

N = 5 * 10**7

def cpu_work(n: int) -> int:
    total = 0
    for i in range(n):
        total += i * i
    return total

async def main():
    loop = asyncio.get_running_loop()
    # Jupyter 里必须显式用 fork；默认 spawn 会 PickleError / 递归
    ctx = multiprocessing.get_context("fork")
    with ProcessPoolExecutor(max_workers=2, mp_context=ctx) as ex:
        results = await asyncio.gather(
            loop.run_in_executor(ex, cpu_work, N),
            loop.run_in_executor(ex, cpu_work, N),
        )
    print(results)

st = time.time()
await main()          # Jupyter cell 顶层可以直接 await
print("time:", time.time() - st)
# 双核机器上约 ~2s；换成 asyncio.to_thread 是 ~4s（GIL 串行）

[41666665416666675000000, 41666665416666675000000]
time: 1.6670069694519043


In [52]:
import timeit

In [56]:
from statistics import median
from time import perf_counter

def benchmark(fn, warmup: int = 5, repeat: int = 20) -> float:
    for _ in range(warmup):
        fn()

    samples = []
    for _ in range(repeat):
        start = perf_counter()
        fn()
        samples.append(perf_counter() - start)
    return median(samples)


In [57]:
import tracemalloc

tracemalloc.start()
# 运行待检查代码
snapshot = tracemalloc.take_snapshot()
for stat in snapshot.statistics("lineno")[:10]:
    print(stat)


/opt/homebrew/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3517: size=296 B, count=1, average=296 B
/opt/homebrew/anaconda3/lib/python3.12/codeop.py:126: size=286 B, count=2, average=143 B
/opt/homebrew/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3577: size=160 B, count=1, average=160 B
/var/folders/_1/bny2r6_91qv_7_6fdv2w9cy80000gn/T/ipykernel_88412/2920723708.py:0: size=72 B, count=1, average=72 B
/opt/homebrew/anaconda3/lib/python3.12/tracemalloc.py:551: size=72 B, count=1, average=72 B
/opt/homebrew/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3507: size=64 B, count=2, average=32 B
/opt/homebrew/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3566: size=64 B, count=1, average=64 B
/opt/homebrew/anaconda3/lib/python3.12/site-packages/IPython/core/compilerop.py:192: size=32 B, count=1, average=32 B
